# TEST Assessment – Diabetes Risk

Clinical assessment prioritizing recall (sensitivity).

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_curve,
    auc
)

sns.set_style("whitegrid")

In [ ]:

# Load test data
X_test = pd.read_parquet("../data/dataset/X_test.parquet")
y_test = pd.read_parquet("../data/dataset/y_test.parquet").squeeze()

# Load trained model
hgb_model = joblib.load("../data/models/HistGradientBoosting.pkl")


In [ ]:
def get_probs(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        df = model.decision_function(X)
        return (df - df.min()) / (df.max() - df.min()) if np.ptp(df) != 0 else np.zeros_like(df)
    return model.predict(X)

def plot_confusion(cm, ax, title, labels=["Negative","Positive"], normalize=False):
    if normalize:
        cm_disp = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_disp, annot=True, fmt=".2f", cmap="Blues", cbar=False,
                    xticklabels=labels, yticklabels=labels, ax=ax)
    else:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)

def find_best_threshold(y_true, y_prob, min_recall=0.85):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    best_t, best_score = None, -1
    best_p, best_r = None, None
    # iterate excluding last precision/recall pair (no threshold)
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        f1 = 2 * (p * r) / (p + r + 1e-9)
        if r >= min_recall and f1 > best_score:
            best_score = f1
            best_t = t
            best_p = p
            best_r = r
        elif best_t is None and f1 > best_score:
            # track best overall in case no threshold reaches min_recall
            best_score = f1
            best_p = p
            best_r = r
    return best_t, best_p, best_r, best_score

models = [(hgb_model, "HistGradientBoosting")]  # ensure these exist
min_recall_target = 0.85


In [ ]:
best_thresholds = {}
summary_rows = []
for model, name in models:
    y_prob = get_probs(model, X_test)
    t, p, r, f1 = find_best_threshold(y_test, y_prob, min_recall=min_recall_target)
    best_thresholds[name] = t
    if t is None:
        note = f"no threshold reaches recall >= {min_recall_target:.2f}; best overall F1 used"
        print(f"{name} → {note}; recall={r:.3f}, precision={p:.3f}, f1={f1:.3f}")
    else:
        print(f"{name} → threshold={t:.3f}, recall={r:.3f}, precision={p:.3f}, f1={f1:.3f}")
    # store for summary table
    summary_rows.append({"model": name, "best_threshold": t if t is not None else np.nan,
                         "precision_at_best": p, "recall_at_best": r, "f1_at_best": f1})

summary_df = pd.DataFrame(summary_rows).set_index("model")
display(summary_df.round(3))


#### Visuals per model

In [ ]:
for model, name in models:
    y_prob = get_probs(model, X_test)
    # PR curve and chosen point
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    # Calibration
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
    # chosen threshold
    t = best_thresholds.get(name)
    if t is None:
        # fallback to threshold that maximizes F1 if none meets min_recall
        f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
        idx = np.nanargmax(f1_scores)
        t = thresholds[idx]
        chosen_p = precision[idx]; chosen_r = recall[idx]; chosen_f1 = f1_scores[idx]
        note = " (fallback best F1)"
    else:
        # find index of threshold closest to t for plotting the point
        idx = np.argmin(np.abs(thresholds - t))
        chosen_p = precision[idx]; chosen_r = recall[idx]; chosen_f1 = 2 * (chosen_p * chosen_r) / (chosen_p + chosen_r + 1e-9)
        note = ""
    print(f"\n{name} summary{note}: threshold={t:.3f}, precision={chosen_p:.3f}, recall={chosen_r:.3f}, f1={chosen_f1:.3f}")
    
    # Plot layout
    fig, axes = plt.subplots(2, 3, figsize=(18,10))
    
    # Confusion matrix (counts)
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    plot_confusion(cm, axes[0,0], f"{name} Confusion Matrix (t={t:.3f})", labels=["Neg","Pos"], normalize=False)
    
    # Confusion matrix (normalized)
    plot_confusion(cm, axes[0,1], f"{name} Confusion Matrix Normalized", labels=["Neg","Pos"], normalize=True)
    
    # Classification report as table
    report = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).T.round(3)
    axes[0,2].axis("off")
    axes[0,2].table(cellText=report_df.values, colLabels=report_df.columns, rowLabels=report_df.index, loc="center")
    axes[0,2].set_title("Classification Report")
    
    # PR curve with chosen point
    axes[1,0].plot(recall, precision, label=f"PR curve (AUC={pr_auc:.3f})")
    axes[1,0].scatter([chosen_r], [chosen_p], color="red", zorder=5, label=f"chosen t={t:.3f}")
    axes[1,0].set_xlabel("Recall")
    axes[1,0].set_ylabel("Precision")
    axes[1,0].set_title("Precision-Recall Curve")
    axes[1,0].legend()
    axes[1,0].grid(alpha=0.3)
    
    # ROC curve
    axes[1,1].plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})")
    axes[1,1].plot([0,1],[0,1], linestyle="--", color="gray")
    axes[1,1].set_xlabel("False Positive Rate")
    axes[1,1].set_ylabel("True Positive Rate")
    axes[1,1].set_title("ROC Curve")
    axes[1,1].legend()
    axes[1,1].grid(alpha=0.3)
    
    # Calibration plot
    axes[1,2].plot(prob_pred, prob_true, marker='o', label="Calibration")
    axes[1,2].plot([0,1],[0,1], linestyle="--", color="gray")
    axes[1,2].set_xlabel("Mean predicted probability")
    axes[1,2].set_ylabel("Fraction of positives")
    axes[1,2].set_title("Calibration Curve")
    axes[1,2].legend()
    axes[1,2].grid(alpha=0.3)
    
    plt.suptitle(f"{name} Detailed Evaluation", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
# --- Metrics across thresholds table ----------------------------------------
thresholds_to_eval = np.linspace(0.0, 1.0, 21)
rows = []
for model, name in models:
    y_prob = get_probs(model, X_test)
    for thr in thresholds_to_eval:
        y_pred = (y_prob >= thr).astype(int)
        p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
        rows.append({"model": name, "threshold": thr, "precision": p, "recall": r, "f1": f})
metrics_by_thr = pd.DataFrame(rows)
display(metrics_by_thr.pivot_table(index="threshold", columns="model", values=["precision","recall","f1"]).round(3))